# D4: Policy Layer & Pareto Analysis

We compare three recalibration policies over a 24-hour simulated day and perform a Pareto analysis across multiple random seeds.

In [ ]:
import sys
sys.path.append('../q-autopilot-backend/src')
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(2026)

In [ ]:
from q_autopilot_backend.sim.policy import simulate_day
from q_autopilot_backend.api.schemas import PolicyId

# Run policies
res_p0 = simulate_day(PolicyId.P0, {}, seed=2026)
res_p1 = simulate_day(PolicyId.P1, {'period_min': 60}, seed=2026)
res_p2 = simulate_day(PolicyId.P2, {'check_interval_min': 5, 'trigger_threshold': 3.0}, seed=2026)

plt.figure(figsize=(10, 5))
plt.plot(res_p0.t_h, res_p0.eps_oracle, label='P0 (Never)')
plt.plot(res_p1.t_h, res_p1.eps_oracle, label='P1 (60 min)')
plt.plot(res_p2.t_h, res_p2.eps_oracle, label='P2 (5min/3\u03c3)')
plt.axhline(1e-3, color='r', linestyle='--', label='Threshold $\epsilon_{th} = 10^{-3}$')
plt.yscale('log')
plt.xlabel('Time of Day (h)')
plt.ylabel('Oracle Gate Error $\epsilon$')
plt.title('Oracle Gate Error vs Time')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from q_autopilot_backend.sim.pareto import compute_pareto
from q_autopilot_backend.api.schemas import UncertaintyType
from q_autopilot_backend.sim.constants import CONFIG

# Use a subset of seeds for faster execution (first 3 seeds)
test_seeds = [CONFIG.OFFICIAL_SEED] + CONFIG.RANDOM_SEEDS[:2]
pareto_points = compute_pareto(UncertaintyType.SD, seeds=test_seeds)

calib_fracs = [p.calib_fraction_mean for p in pareto_points]
eps_means = [p.mean_eps_mean for p in pareto_points]
calib_errs = [p.calib_fraction_err for p in pareto_points]
eps_errs = [p.mean_eps_err for p in pareto_points]

plt.figure(figsize=(8, 6))
plt.errorbar(calib_fracs, eps_means, xerr=calib_errs, yerr=eps_errs, fmt='o', alpha=0.6)
plt.xlabel('Fraction of day spent calibrating')
plt.ylabel('Time-averaged Oracle Gate Error')
plt.title('Pareto Frontier (3 Random Seeds - Fast Test)')
plt.yscale('log')
plt.grid(True)
plt.show()